# Notebook 04 — SARIMAX con variables exógenas

Este notebook extiende el modelo SARIMA incorporando variables exógenas
para mejorar la predicción de la huella de carbono eléctrica.

Las variables exógenas se agrupan en dos categorías con justificación física:
- **Generación renovable**: solar, eólica, hidráulica, nuclear, geotérmica, marina
- **Generación no renovable**: gas, carbón, petróleo, biomasa, residuos, otros

Dado que las variables exógenas futuras no son conocidas en el momento
de la predicción, se estiman mediante un modelo Naive_seasonal antes
de introducirlas en el SARIMAX.

Posición del sol, seno y coseno de la hora, velocidad del viento

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

plt.style.use("seaborn-v0_8")
%matplotlib inline

## Carga de datos de huella de carbono

Se cargan los conjuntos de entrenamiento, validación y test
ya procesados en el notebook anterior.

In [4]:
DATA_DIR = Path("data_processed")

y_train = pd.read_parquet(DATA_DIR / "train_2022_2023.parquet")["y"].astype(float)
y_val   = pd.read_parquet(DATA_DIR / "val_2024.parquet")["y"].astype(float)
y_test  = pd.read_parquet(DATA_DIR / "test_2025.parquet")["y"].astype(float)

y_train.index = pd.to_datetime(y_train.index)
y_val.index   = pd.to_datetime(y_val.index)
y_test.index  = pd.to_datetime(y_test.index)

print("Train:", y_train.shape, "|", y_train.index.min(), "->", y_train.index.max())
print("Val:  ", y_val.shape,   "|", y_val.index.min(),   "->", y_val.index.max())
print("Test: ", y_test.shape,  "|", y_test.index.min(),  "->", y_test.index.max())

Train: (70080,) | 2022-01-01 00:00:00+00:00 -> 2023-12-31 23:45:00+00:00
Val:   (35136,) | 2024-01-01 00:00:00+00:00 -> 2024-12-31 23:45:00+00:00
Test:  (35040,) | 2025-01-01 00:00:00+00:00 -> 2025-12-31 23:45:00+00:00


## Carga y construcción de variables exógenas

Se cargan los ficheros de generación por tecnología para España (columna ES)
para los años 2022 a 2025, concatenando los ficheros anuales.

Las tecnologías se agrupan en:
- **Renovable**: solar, wind_onshore, wind_offshore, hydro_river,
  hydro_reservoir, nuclear, geothermal, marine
- **No renovable**: gas, coal, oil, biomass, waste, other

In [ ]:
RAW_DIR = Path("../data_raw")  

RENEWABLES = [
    "solar_generation",
    "wind_onshore_generation",
    "wind_offshore_generation",
    "hydro_river_generation",
    "hydro_reservoir_generation",
    "nuclear_generation",
    "geothermal_generation",
    "marine_generation"
]

NON_RENEWABLES = [
    "gas_generation",
    "coal_generation",
    "oil_generation",
    "biomass_generation",
    "waste_generation",
    "other_generation"
]

YEARS = [2022, 2023, 2024, 2025]

def load_generation(tech_name, years, country="ES"):
    """Carga y concatena un fichero de generación para todos los años."""
    dfs = []
    for year in years:
        path = RAW_DIR / str(year) / "generation" / f"{tech_name}.csv"
        df = pd.read_csv(path, index_col="timestamp", parse_dates=True)
        dfs.append(df[[country]])
    return pd.concat(dfs).sort_index()[country].rename(tech_name)

# Cargar todas las tecnologías
all_series = {}
for tech in RENEWABLES + NON_RENEWABLES:
    all_series[tech] = load_generation(tech, YEARS)

# Construir dataframe de exógenas
exog_df = pd.DataFrame(all_series)

# Agrupar
exog_df["renewable"]     = exog_df[RENEWABLES].sum(axis=1)
exog_df["non_renewable"] = exog_df[NON_RENEWABLES].sum(axis=1)

# Quedarse solo con las dos variables agrupadas
exog_df = exog_df[["renewable", "non_renewable"]]

print(exog_df.shape)
exog_df.head()

(140256, 2)


,renewable,non_renewable
timestamp,,
2022-01-01 00:00:00+00:00,14993.0,5693.0
2022-01-01 00:15:00+00:00,14993.0,5693.0
2022-01-01 00:30:00+00:00,14993.0,5693.0
2022-01-01 00:45:00+00:00,14993.0,5693.0
2022-01-01 01:00:00+00:00,14415.0,5350.0


## División temporal de las variables exógenas

Se aplica la misma división temporal que en la variable objetivo:
- Train: 2022–2023
- Validación: 2024
- Test: 2025

In [7]:
exog_train = exog_df.loc["2022":"2023"]
exog_val   = exog_df.loc["2024"]
exog_test  = exog_df.loc["2025"]

print("Exog train:", exog_train.shape)
print("Exog val:  ", exog_val.shape)
print("Exog test: ", exog_test.shape)

# Verificar alineación con variable objetivo
print("\nAlineación con y_train:", exog_train.index.equals(y_train.index))
print("Alineación con y_val:  ", exog_val.index.equals(y_val.index))

Exog train: (70080, 2)
Exog val:   (35136, 2)
Exog test:  (35040, 2)

Alineación con y_train: True
Alineación con y_val:   True


## Predicción de variables exógenas futuras

Dado que SARIMAX requiere conocer los valores futuros de las variables
exógenas en el momento de la predicción, se estiman mediante un modelo
Naive_seasonal (lag=96), que replica el patrón del día anterior.

Esta es una limitación conocida del enfoque: el error del SARIMAX
acumula el error de predicción de las exógenas más el error propio
del modelo.

In [8]:
def forecast_exog_naive(exog_series, start_idx, horizon, season=96):
    """
    Predice variables exógenas usando Naive_seasonal (lag=96).
    Replica el patrón del último día disponible hasta cubrir el horizonte.
    """
    last_season = exog_series.iloc[start_idx - season:start_idx]
    reps = int(np.ceil(horizon / season))
    forecast = np.tile(last_season.values, (reps, 1))[:horizon]
    return pd.DataFrame(
        forecast,
        columns=exog_series.columns
    )

# Test rápido
exog_full = pd.concat([exog_train, exog_val])
test_exog_pred = forecast_exog_naive(exog_full, len(exog_train), 192)
print("Shape predicción exógenas:", test_exog_pred.shape)
test_exog_pred.head()

Shape predicción exógenas: (192, 2)


,renewable,non_renewable
0,21968.0,3776.0
1,22108.0,4124.0
2,22024.0,4208.0
3,21812.0,4040.0
4,21548.0,3888.0


## Configuración temporal y métricas

Se reutilizan los mismos parámetros temporales y métricas del notebook anterior.

In [11]:
FREQ_MIN = 15
STEPS_PER_HOUR = 60 // FREQ_MIN
SEASONAL_PERIOD = 24 * STEPS_PER_HOUR  # 96

HORIZONS = {
    "48h": 48 * STEPS_PER_HOUR,
    "72h": 72 * STEPS_PER_HOUR
}

from dataclasses import dataclass

@dataclass
class WFConfig:
    step: int
    min_history: int
    max_fits: int

def compute_metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {"MAE": mae, "RMSE": rmse}

## Modelo SARIMAX

Se emplea el mismo orden identificado en el notebook anterior:
SARIMA(2,1,0)(1,1,0,96), extendido con las variables exógenas
renewable y non_renewable.

Las variables exógenas futuras se predicen mediante Naive_seasonal
antes de introducirlas en el modelo.

In [12]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_order          = (2, 1, 0)
sarima_seasonal_order = (1, 1, 0, 96)

def forecast_sarimax(y_train, exog_train, exog_future, h,
                     order, seasonal_order, maxiter=20):
    """
    Entrena SARIMAX con variables exógenas conocidas (train)
    y predice usando las exógenas futuras estimadas.
    """
    model = SARIMAX(
        y_train.values,
        exog=exog_train.values,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    res = model.fit(disp=False, maxiter=maxiter, method="lbfgs")
    return res.forecast(steps=h, exog=exog_future.values)

## Evaluación walk-forward SARIMAX

Se usa la misma configuración reducida que en SARIMA para
mantener el coste computacional asumible.

In [13]:
from dataclasses import dataclass

cfg_sarimax = WFConfig(
    step=7 * SEASONAL_PERIOD,
    min_history=14 * SEASONAL_PERIOD,
    max_fits=10
)

def walk_forward_splits(series, horizons, cfg):
    n     = len(series)
    max_h = max(horizons.values())
    points = list(range(cfg.min_history, n - max_h, cfg.step))
    points = points[:cfg.max_fits]
    for t0 in points:
        train_part = series.iloc[:t0]
        tests = {k: series.iloc[t0:t0 + h] for k, h in horizons.items()}
        yield train_part, tests

def evaluate_sarimax(y_train, y_val, exog_train, exog_val,
                     horizons, cfg, order, seasonal_order,
                     train_window_days=14, verbose=True):
    rows   = []
    exog_full = pd.concat([exog_train, exog_val])
    y_full    = pd.concat([y_train, y_val])

    splits = list(walk_forward_splits(y_val, horizons, cfg))
    total  = len(splits)

    for i, (train_part, tests) in enumerate(splits, 1):
        if verbose:
            print(f"Fit {i}/{total}")

        # Historia disponible
        y_hist    = pd.concat([y_train, train_part]).sort_index()
        exog_hist = pd.concat([exog_train, exog_val.iloc[:len(train_part)]]).sort_index()

        # Ventana fija
        cutoff    = y_hist.index.max() - pd.Timedelta(days=train_window_days)
        y_hist    = y_hist.loc[y_hist.index >= cutoff]
        exog_hist = exog_hist.loc[exog_hist.index >= cutoff]

        for name, test in tests.items():
            h = len(test)

            # Predecir exógenas futuras con Naive_seasonal
            exog_future = forecast_exog_naive(
                exog_full,
                len(pd.concat([exog_train, exog_val.iloc[:len(train_part)]])),
                h
            )

            pred = forecast_sarimax(
                y_hist, exog_hist, exog_future,
                h, order, seasonal_order
            )

            m = compute_metrics(test.values, pred)
            rows.append({"horizon": name, "MAE": m["MAE"], "RMSE": m["RMSE"]})

    return pd.DataFrame(rows)

## Resultados SARIMAX

In [14]:
sarimax_val = evaluate_sarimax(
    y_train, y_val,
    exog_train, exog_val,
    HORIZONS,
    cfg_sarimax,
    sarima_order,
    sarima_seasonal_order,
    train_window_days=14
)

print("Filas generadas:", len(sarimax_val))
sarimax_val.head()

Fit 1/10


KeyboardInterrupt: 

In [ ]:
RESULTS_DIR = Path("../results")

naive_summary  = pd.read_csv(RESULTS_DIR / "naive_summary.csv")
arima_summary  = pd.read_csv(RESULTS_DIR / "arima_summary.csv")
sarima_summary = pd.read_csv(RESULTS_DIR / "sarima_summary.csv")

In [ ]:
def summarize(df, model_name):
    out = (
        df.groupby("horizon")[["MAE", "RMSE"]]
        .mean()
        .reset_index()
    )
    out.insert(0, "model", model_name)
    return out

sarimax_summary = summarize(sarimax_val, "SARIMAX(2,1,0)(1,1,0,96)")
sarimax_summary

In [ ]:
compare_val = pd.concat([
    naive_summary,
    arima_summary,
    sarima_summary,
    sarimax_summary
], ignore_index=True)